

---

---




# **CROSS DOMAIN TRAFFIC SIGN DETECTION**
### The goal of the project is to solve Domain Shift—when an AI fails because the training data looks different from the testing data.

---

---


```

                    ARCHITECTURE OF THE PROJECT
                -----------------------------------

+---------------------------------------------------------------+
|       CROSS-DOMAIN TRAFFIC SIGN DETECTION ARCHITECTURE        |
+---------------------------------------------------------------+
                                |
                                V
+---------------------------------------------------------------+
|              PHASE 0: ENVIRONMENT INITIALIZATION              |
+---------------------------------------------------------------+
|  [1] Mount Google Drive (/content/drive)                      |
|                               |                               |
|                               V                               |
|  [2] Set Working Directory (os.chdir)                         |
|      (Routes all downloads, weights, and logs directly to     |
|      the 'CrossDomainTraffic' folder in Google Drive)         |
+---------------------------------------------------------------+
                                |
                                V
+---------------------------------------------------------------+
|          PHASE 1: DATA PIPELINE (SOURCE & TARGET)             |
+---------------------------------------------------------------+
|  [1] Download Datasets via Roboflow API directly to Drive     |
|      - Source: Asian/US Road Signs (roboflow-100)             |
|      - Target: German Traffic Signs (GTSRB)                   |
|                               |                               |
|                               V                               |
|  [2] Inspect & Visualize Data                                 |
|      (Calculate disk size, image counts, display 10 samples)  |
+---------------------------------------------------------------+
                                |
                                V
+---------------------------------------------------------------+
|       PHASE 2: DATA & CONFIGURATION ALIGNMENT (CRITICAL)      |
+---------------------------------------------------------------+
|  [1] Repair File Paths in Google Drive                        |
|      (Update data.yaml paths for absolute Colab/Drive paths)  |
|                               |                               |
|                               V                               |
|  [2] Prevent IndexError (Label Alignment)                     |
|      (Scan Target .txt labels and filter classes >= Source)   |
|                               |                               |
|                               V                               |
|  [3] Synchronize Metadata                                     |
|      (Force Target YAML 'nc' and 'names' to match Source)     |
+---------------------------------------------------------------+
                                |
                                V
+---------------------------------------------------------------+
|              PHASE 3: BASELINE MODEL TRAINING                 |
+---------------------------------------------------------------+
|  [1] Initialize Base Model                                    |
|      (Load YOLOv8 Nano - yolov8n.pt)                          |
|                               |                               |
|                               V                               |
|  [2] Train Baseline Model                                     |
|      (Train exclusively on Source Data for 15 epochs)         |
|                               |                               |
|                               V                               |
|  [3] Save Baseline Weights to Drive                           |
|      (traffic_project/source_model/weights/best.pt)           |
+---------------------------------------------------------------+
                                |
                                V
+---------------------------------------------------------------+
|              PHASE 4: EVALUATE DOMAIN GAP                     |
+---------------------------------------------------------------+
|  [1] Load Baseline Weights                                    |
|                               |                               |
|                               V                               |
|  [2] Validate Baseline Model on Target Data (German)          |
|                               |                               |
|                               V                               |
|  [3] Record Baseline Accuracy (mAP@50)                        |
|      (Quantifies the "Domain Shift" problem mathematically    |
|       via zero-shot testing)                                  |
+---------------------------------------------------------------+
                                |
                                V
+---------------------------------------------------------------+
|       PHASE 5: DOMAIN ADAPTATION (TRANSFER LEARNING)          |
+---------------------------------------------------------------+
|  [1] Initialize Adaptation Model                              |
|      (Load Baseline weights trained on Source)                |
|                               |                               |
|                               V                               |
|  [2] Fine-Tune on Target Data                                 |
|      (Train on German data with low Learning Rate: 0.005      |
|       to prevent Catastrophic Forgetting)                     |
|                               |                               |
|                               V                               |
|  [3] Save Adapted Weights to Drive                            |
|      (traffic_project/adapted_model/weights/best.pt)          |
+---------------------------------------------------------------+
                                |
                                V
+---------------------------------------------------------------+
|             PHASE 6: FINAL EVALUATION & REPORTING             |
+---------------------------------------------------------------+
|  [1] Validate Adapted Model on Target Data                    |
|                               |                               |
|                               V                               |
|  [2] Calculate Metrics                                        |
|      (Compare Final mAP vs Baseline mAP for gap reduction)    |
|                               |                               |
|                               V                               |
|  [3] Export Project Report to Drive                           |
|      (Save 'Final_Project_Report.txt' permanently)            |
|                               |                               |
|                               V                               |
|  [4] Plot Confusion Matrix                                    |
|      (Read and visualize image directly from Drive)           |
+---------------------------------------------------------------+

```








---


### **INSTALLATIONS**

---



In [ ]:
!pip install ultralytics roboflow pandas matplotlib seaborn tqdm -q

In [ ]:
import torch
import ultralytics
from ultralytics import YOLO
import os
import sys
import yaml
import glob
import random
from IPython.display import Image, display
import matplotlib.pyplot as plt

# check GPU status
print(f"PyTorch version: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU is available: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available. Go to Runtime -> Change runtime type -> T4 GPU")

# verify YOLO is ready
ultralytics.checks()



---
### **AUTHENTICATION**

---





In [ ]:
from google.colab import drive
import os

print("Connecting to Google Drive...")
drive.mount('/content/drive')

base_dir = '/content/drive/MyDrive/Colab_Notebooks/CrossDomainTraffic'

# if there is no folder following will create automatically
os.makedirs(base_dir, exist_ok=True)

# changing current working directory
os.chdir(base_dir)

print(f"\nSuccess! current working directory is now set to: {os.getcwd()}")


#### STEP 1 (a)
---


### **DOWNLOAD DATASETS (SOURCE VS TARGET)**

---




Here we download two different datasets to simulate the "Domain Shift."
*   Source: A generic road sign dataset (acts as our "Synthetic/Training" domain).

    *Link: https://universe.roboflow.com/roboflow-100/road-signs-6ih4y*

*   Target: The famous GTSRB (German Traffic Sign) dataset (acts as our "Real World" domain).

    *Link: https://universe.roboflow.com/gtsrbanno/traffic-sign-detection-gtsrb*



In [ ]:
from roboflow import Roboflow
import sys

API_KEY = "5gf0ocpYdwrY8xNSXV6w"

rf = Roboflow(api_key=API_KEY)

# download Source Data (Generic/US)
print("Downloading source data (Generic/US)...")
try:
    project_source = rf.workspace("roboflow-100").project("road-signs-6ih4y")
    dataset_source = project_source.version(1).download("yolov8")
except Exception as e:
    print(f"\nERROR Downloading Source: {e}")
    sys.exit("Stopping execution. Please check your API Key.")

# download Target Data (real German GTSRB)
# swapped to 'gtsrbanno' workspace which is stable and public
print("\nDownloading target Data (German GTSRB)...")
try:
    project_target = rf.workspace("gtsrbanno").project("traffic-sign-detection-gtsrb")
    dataset_target = project_target.version(1).download("yolov8")
except Exception as e:
    print(f"\nError downloading Target: {e}")
    sys.exit("Stopping execution. Please check your API Key.")

# define paths
if dataset_source and dataset_target:
    source_yaml = f"{dataset_source.location}/data.yaml"
    target_yaml = f"{dataset_target.location}/data.yaml"
    print("\nDatasets downloaded successfully!")
else:
    print("\nOne of the datasets failed to load.")

#### 1 (b)

---


### **DATASETS DETAILS**

---



In [ ]:
import yaml
import glob
import os

def get_dir_size(start_path='.'):
    """Calculates total size of a directory in Megabytes (MB)."""
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(start_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            # skip if it is symbolic link
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024) # convert bytes to MB

def get_dataset_stats(yaml_path, dataset_name):
    """Reads the yaml file, counts images, and calculates disk size."""

    # load the configuration
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    # get base directory
    base_dir = os.path.dirname(yaml_path)

    # count images
    train_count = len(glob.glob(f"{base_dir}/train/images/*.jpg"))
    val_count = len(glob.glob(f"{base_dir}/valid/images/*.jpg"))
    test_count = len(glob.glob(f"{base_dir}/test/images/*.jpg"))

    # calculate size
    size_mb = get_dir_size(base_dir)

    # print report
    print(f"==========================================")
    print(f"DATASET REPORT: {dataset_name}")
    print(f"==========================================")
    print(f"Path:        {base_dir}")
    print(f"Disk Size:   {size_mb:.2f} MB")
    print(f"Class Count: {data.get('nc', 'Unknown')}")
    print(f"Class Names: {data.get('names', [])[:5]} ... (showing first 5)")
    print(f"")
    print(f"IMAGE COUNTS:")
    print(f"Training Images:   {train_count}")
    print(f"Validation Images: {val_count}")
    print(f"Test Images:       {test_count}")
    print(f"TOTAL IMAGES:      {train_count + val_count + test_count}")
    print(f"\n")

# run the inspector for both
print("INSPECTING DATASETS & SIZES...\n")

if 'source_yaml' in locals() and 'target_yaml' in locals():
    get_dataset_stats(source_yaml, "SOURCE (Generic/US)")
    get_dataset_stats(target_yaml, "TARGET (German GTSRB)")
else:
    print("Error: Variables 'source_yaml' and 'target_yaml' not found.")
    print("Please make sure you have run Step 2 (Download) first.")

#### 1 (c)

---


### **SHOWING 10 RANDOM IMAGES OF BOTH THE DATASETS**

---
This is a Cross-Geography Domain Adaptation project. The model is trained on an Asian/International dataset (Source) and adapted it to the German GTSRB standard (Target). This simulates the real-world business problem of training a self-driving car in one country and shipping it to another, which creates a significant Domain Shift challenge similar to Synthetic-to-Real adaptation.


In [ ]:
import glob
import random
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import os

def display_random_10_images(yaml_path, title):
    """
    Finds the training images folder and displays
    10 random images in a grid.
    """
    # construct the path to training images
    base_dir = os.path.dirname(yaml_path)
    # check common paths
    img_dir = os.path.join(base_dir, "train", "images")
    if not os.path.exists(img_dir):
        img_dir = os.path.join(base_dir, "train")

    # get full list of images
    image_files = glob.glob(f"{img_dir}/*.jpg") + glob.glob(f"{img_dir}/*.png")

    if not image_files:
        print(f"No images found in {img_dir}")
        return

    # pick Random 10
    num_to_show = min(10, len(image_files))
    selected_files = random.sample(image_files, num_to_show)

    print(f"\n{title} (Random 10 Samples):")
    print(f"   Source Path: {img_dir}")

    # Setup the Plot (2 rows of 5 images)
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    axes = axes.flatten()

    for i, ax in enumerate(axes):
        if i < len(selected_files):
            try:
                img = mpimg.imread(selected_files[i])
                ax.imshow(img)
                ax.set_title(f"Sample {i+1}", fontsize=9)
                ax.axis('off')
            except Exception as e:
                ax.text(0.5, 0.5, "Error loading", ha='center')
                ax.axis('off')
        else:
            ax.axis('off') # hide empty subplots

    plt.tight_layout()
    plt.show()

# execute display
if 'source_yaml' in locals() and 'target_yaml' in locals():
    display_random_10_images(source_yaml, "SOURCE DATASET (Generic/US)")
    display_random_10_images(target_yaml, "TARGET DATASET (German GTSRB)")
else:
    print("Error: Variables 'source_yaml' and 'target_yaml' not found. Run Step 2 first.")

#### STEP 2

---


### **DATA ALLIGNMENT**

---



*   It fixes the FileNotFoundError (paths) and the IndexError (class mismatch) automatically.

*  We programmatically filtered the ground-truth label files in the Target dataset to exclude any classes that did not exist in the Source domain. This prevented runtime index errors during inference.

*   This script scanned the actual *.txt* label files in the dataset. It removed any object labels with a Class ID of 22 or higher.
*   Without this, the model would crash during testing (Runtime IndexError). The model's final output layer is a vector of size 21. If the data loader feeds it a "Class 42" label, the model tries to check the 42nd index of a 21-length vector, causing a crash.




In [ ]:
# automatic data repair
print("Running Automatic Dataset Repair...")

def fix_yaml_paths(yaml_path):
    """Repairs file paths in data.yaml to work in Colab"""
    base_dir = os.path.dirname(yaml_path)
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    # set absolute paths
    data['train'] = os.path.join(base_dir, "train/images")
    data['val'] = os.path.join(base_dir, "valid/images")

    test_path = os.path.join(base_dir, "test/images")
    if os.path.exists(test_path):
        data['test'] = test_path
    else:
        data['test'] = data['val'] # fallback if no test folder

    # remove conflicting 'path' key
    if 'path' in data: del data['path']

    with open(yaml_path, 'w') as f:
        yaml.dump(data, f)
    return data

# fix Paths for both datasets
fix_yaml_paths(source_yaml)
target_data = fix_yaml_paths(target_yaml)
print("File paths repaired.")

# align Classes (prevent indexError)
# the German dataset has more classes than the US one. We must remove the extra ones.
with open(source_yaml, 'r') as f:
    source_nc = yaml.safe_load(f)['nc'] # get number of classes in source

# update target Config to match Source
target_data['nc'] = source_nc
with open(target_yaml, 'w') as f:
    yaml.dump(target_data, f)

# scan all label files and remove lines with Class ID >= source_nc
print(f"Removing classes >= {source_nc} from Target dataset...")
removed_count = 0
target_dir = os.path.dirname(target_yaml)

for folder in ['train', 'valid', 'test']:
    label_files = glob.glob(f"{target_dir}/{folder}/labels/*.txt")
    for file in label_files:
        with open(file, 'r') as f:
            lines = f.readlines()

        # keep only lines where class_id is small enough
        new_lines = []
        for line in lines:
            parts = line.split()
            if parts and int(parts[0]) < source_nc:
                new_lines.append(line)

        # overwrite file if changes were made
        if len(new_lines) != len(lines):
            removed_count += (len(lines) - len(new_lines))
            with open(file, 'w') as f:
                f.writelines(new_lines)

print(f"Classes aligned. Removed {removed_count} incompatible labels.")
print("DATASET READY FOR TRAINING.")

In [ ]:
import yaml
import glob
import os

def get_dir_size(start_path='.'):
    """Calculates total size of a directory in Megabytes (MB)."""
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(start_path):
        for f in filenames:
            fp = os.path.join(dirpath, f)
            # skip if it is symbolic link
            if not os.path.islink(fp):
                total_size += os.path.getsize(fp)
    return total_size / (1024 * 1024) # convert bytes to MB

def get_dataset_stats(yaml_path, dataset_name):
    """Reads the yaml file, counts images, and calculates disk size."""

    # load the Configuration
    with open(yaml_path, 'r') as f:
        data = yaml.safe_load(f)

    # get Base Directory
    base_dir = os.path.dirname(yaml_path)

    # count images
    train_count = len(glob.glob(f"{base_dir}/train/images/*.jpg"))
    val_count = len(glob.glob(f"{base_dir}/valid/images/*.jpg"))
    test_count = len(glob.glob(f"{base_dir}/test/images/*.jpg"))

    # calculate size
    size_mb = get_dir_size(base_dir)

    # print report
    print(f"==========================================")
    print(f"DATASET REPORT: {dataset_name}")
    print(f"==========================================")
    print(f"Path:        {base_dir}")
    print(f"Disk Size:   {size_mb:.2f} MB")
    print(f"Class Count: {data.get('nc', 'Unknown')}")
    print(f"Class Names: {data.get('names', [])[:5]} ... (showing first 5)")
    print(f"")
    print(f"IMAGE COUNTS:")
    print(f"Training Images:   {train_count}")
    print(f"Validation Images: {val_count}")
    print(f"Test Images:       {test_count}")
    print(f"TOTAL IMAGES:      {train_count + val_count + test_count}")
    print(f"\n")

# run the inspector for both
print("INSPECTING DATASETS & SIZES...\n")

if 'source_yaml' in locals() and 'target_yaml' in locals():
    get_dataset_stats(source_yaml, "SOURCE (Generic/US)")
    get_dataset_stats(target_yaml, "TARGET (German GTSRB)")
else:
    print("Error: variables 'source_yaml' and 'target_yaml' not found.")
    print("Please make sure you have run Step 2 (download) first.")


#### STEP 3
---


### **TRAIN BASELINE MODEL (ON SOURCE DOMAIN)**

---




*   We train a fresh YOLOv8 model only on the Source data. This simulates training an AI on a simulator or a different country's data.





In [ ]:
# initialize a fresh YOLOv8 nano model (smallest & fastest)
model = YOLO('yolov8n.pt')

print("Starting training on source domain...")
# we train for 15 epochs to ensure it learns the patterns.
model.train(
    data=source_yaml,
    epochs=15,
    imgsz=640,
    batch=16,
    project='traffic_project',
    name='source_model',
    device=0,  # use the GPU (0)
    verbose=True
)
print("Source training finished.")

#### STEP 4

---


### **CONFIGURATION ALLIGNMENT**

---




*   We synchronized the data.yaml metadata file to match the Source model's schema. This ensured the YOLO validator accepted the dataset structure during initialization.
*   This script updated the data.yaml configuration file. It overwrote the German dataset's class list (47 classes) with the Source dataset's class list (21 classes).

*  Without this, the YOLO framework would refuse to even start the evaluation (Initialization SyntaxError). Before loading any images, YOLO checks if the model's structure (nc: 21) matches the dataset's declared structure (nc: 47). If they differ, it blocks execution to prevent logic errors.

In [ ]:
import yaml

print("Fixing config mismatch...")

# load the source config (the "correct" reference)
with open(source_yaml, 'r') as f:
    source_data = yaml.safe_load(f)
    correct_names = source_data['names']
    correct_nc = source_data['nc']

# load the target config (the "broken" one)
with open(target_yaml, 'r') as f:
    target_data = yaml.safe_load(f)

print(f"Found mismatch: target says nc={target_data['nc']} but has {len(target_data['names'])} names.")

# force target to match source exactly
target_data['nc'] = correct_nc
target_data['names'] = correct_names

# save the fixed file
with open(target_yaml, 'w') as f:
    yaml.dump(target_data, f)

print(f"Fixed: Target now has nc={correct_nc} and {len(correct_names)} names.")

#### STEP 5

---


### **EVALUATE DOMAIN GAP (THE PROBLEM)**

---




*   Now we test that model on the German (Target) signs. It should perform poorly because it hasn't seen German signs yet. This proves the need for our project.



In [ ]:
print("Testing source model on target (German) Data...")

# load weights from the previous step
source_weights = "/content/drive/MyDrive/Colab_Notebooks/CrossDomainTraffic/runs/detect/traffic_project/source_model/weights/best.pt"
model_eval = YOLO(source_weights)

# validate on target dataset
metrics_baseline = model_eval.val(data=target_yaml, split='test', project='traffic_project', name='eval_baseline')

baseline_map = metrics_baseline.box.map50
print(f"\nBaseline accuracy (mAP@50): {baseline_map:.4f}")


#### STEP 6
---


### **DOMAIN ADAPTATION (THE SOLUTION)**

---




*   We now use Transfer Learning. We take the model that knows generic signs and "fine-tune" it on the German data.
*   We use a low learning rate (lr0 = 0.005). This gently updates the model without erasing what it already knows.



In [ ]:
print("Applying domain adaptation (Fine-Tuning)...")

# load the source model again
model_adapt = YOLO(source_weights)

# train it briefly on the target dataset
model_adapt.train(
    data=target_yaml,
    epochs=15,            # short training is enough for adaptation
    imgsz=640,
    batch=16,
    lr0=0.005,            # low learning rate = gentle adaptation
    project='traffic_project',
    name='adapted_model',
    device=0
)
print("Adaptation complete!")


#### STEP 7
---


### **FINAL EVALUATION AND DELIVERABLES**

---



In [ ]:
print("Testing adapted model on target data...")
adapted_weights = '/content/drive/MyDrive/Colab_Notebooks/CrossDomainTraffic/runs/detect/traffic_project/adapted_model/weights/best.pt'
final_model = YOLO(adapted_weights)

metrics_final = final_model.val(data=target_yaml, split='test', project='traffic_project', name='eval_final')
final_map = metrics_final.box.map50

# calculate gap reduction
improvement = final_map - baseline_map
pct_increase = (improvement / baseline_map) * 100 if baseline_map > 0 else 0

# generate comparative report
report_text = f"""
======================================================
       PROJECT: CROSS-DOMAIN TRAFFIC DETECTION
       Comparative Analysis Report
======================================================
1. Strategy Used:      Transfer Learning (Fine-Tuning)
2. Source Data:        Generic/US Road Signs
3. Target Data:        German Traffic Signs (GTSRB)

--- PERFORMANCE METRICS (mAP@0.50) ---
[A] Baseline Model:    {baseline_map:.4f} (Trained only on Source)
[B] Adapted Model:     {final_map:.4f} (After Domain Adaptation)

--- RESULTS ---
>> Domain Gap Reduction: +{improvement:.4f} mAP
>> Percentage Increase:  {pct_increase:.1f}%
======================================================
"""

print(report_text)

# save report to a text file (deliverable)
with open("Final_Project_Report.txt", "w") as f:
    f.write(report_text)


#### STEP 8
---


### **CONFUSION MATRIX**

---

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

print("--- Confusion Matrix ---")
cm_path = '/content/drive/MyDrive/Colab_Notebooks/CrossDomainTraffic/runs/detect/traffic_project/adapted_model/confusion_matrix.png'

if os.path.exists(cm_path):
    # this method embeds the image data permanently into the notebook
    img = mpimg.imread(cm_path)
    plt.figure(figsize=(10, 10)) # adjust size as needed
    plt.imshow(img)
    plt.axis('off') # hides the X and Y axis numbers
    plt.show()
else:
    print("Error: Confusion matrix image not found at path:", cm_path)